---

digital brain MEG pipeline

---

In [ ]:
# autoload
%load_ext autoreload
%autoreload 2

# load pgl actions
from pgl import pglActions, pglActionsMNE, pglStorage, pgl, pglSession
pgl = pgl()

---

Help on all actions available

---

In [ ]:
pglActionsMNE.help()
#pglActionsMNE.help("loadFieldline")

---

load session

---

In [ ]:

# Load the session
#loadSession = pglActions.loadSession()
#loadSession.configure()
#if loadSession: session = loadSession.run()

---

Load FieldLine data

---

In [ ]:
# execute load
session = pglActionsMNE.loadFieldline.execute()

In [ ]:
session = pglSession.loadSession(55)

In [ ]:
pglStorage.print(sessionID=61)

In [ ]:
session = pglSession.loadSession(sessionID=61)

In [ ]:
session.mne.displayReport()

---

Concatenate raws

---

In [ ]:
# configure the concatenation
newSession = pglActionsMNE.concatenate.execute(session=session, all=True)
if newSession: session = newSession

---

Filter data

---

In [ ]:
session = pglActionsMNE.filter.execute(session=session, lowCutoff=0.5, highCutoff=50.0, notch=True, notchFrequency=60.0)
if newSession: session = newSession

---

Bads handling

---

In [ ]:
session = pglActionsMNE.bads.execute(session=session, excludeWhenPositionIsNaN=True, extraBads=[], interpolationOrigin="zero", method="interpolate")
if newSession: session = newSession

---

Configure events (for epoching), sets names of conditions

---

In [ ]:
# group all the things labels into one
grouped = {
    'things': range(2,201),
    'blank': 1022,
    'catch': 1023
}

# label each stimulus with its own name
each = {
    f"stimulus {code:03d}": code
    for code in range(2, 201)
}
each["blank"] = 1022
each["catch"] = 1023

triggerLabelSets = {
    "grouped": grouped,
    "each": each
}

# configure the events
newSession = pglActionsMNE.configureEvents.execute(session=session, triggerChannel='di2', triggerShortestEvent=1, triggerLabelSets = triggerLabelSets)
if newSession: session = newSession

---

create epochs

---

In [ ]:
newSession = pglActionsMNE.createEpochs.execute(session=session, tmin=0.0, tmax=2.0)
if newSession: session = newSession

---

Mark interruns as bad

---

In [ ]:
newSession = pglActionsMNE.markInterRunsAsBad.execute(session=session, tmin=0.0, tmax=2.0)
if newSession: session = newSession

---

Plot evoked

---

In [ ]:
plotEvoked = pglActionsMNE.plotEvoked()
#plotEvoked.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks="L213")
#plotEvoked.configure(set="each",label="stimulus 010")
plotEvoked.configure(set="grouped",label="things",minFreq=0,maxFreq=20)

In [ ]:
newSession = plotEvoked.run(session)
if newSession: session = newSession

---

Eyeblink rejection

---

In [ ]:
# fit and display ICA components
newSession = pglActionsMNE.ica.execute(session=session, chType='mag',nComponents=20)
if newSession: session = newSession

In [ ]:
# mark as bad based on ICA component from above and threshold in std
newSession = pglActionsMNE.markBlinks.execute(session=session, component=8, threshold=2)
if newSession: session = newSession

In [ ]:
# Create epochs, respecting the bads from the eyeblink (will fill time periods with NaN)
newSession = pglActionsMNE.createEpochsWithBads.execute(session=session)
if newSession: session = newSession

---

Plot evoked respecting Nans introduced by blink detection

---

In [ ]:
plotEvokedNan = pglActionsMNE.plotEvokedNan()
#plotEvokedNan.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks="L213")
#plotEvokedNan.configure(set="each",label="stimulus 010",picks='eeg')
#plotEvokedNan.configure(set="grouped",label="catch",minFreq=0,maxFreq=20,picks='eeg')
plotEvokedNan.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks='mag')
#mnePlotEvoked.configure(set="each",label="stimulus 021",minFreq=0,maxFreq=20,picks='eeg')

In [ ]:
newSession = plotEvokedNan.run(session=session)

---

Plot evoked each NaN version

---

In [ ]:
newSession = pglActionsMNE.plotEvokedEachNan.execute(session=session, set="each", picks='mag',topoMin =-1500, topoMax=1500, timeseriesMin=-1500, timeseriesMax=1500, topoTime=284)
if newSession: session = newSession

---

Downsample epochs

---

In [ ]:
newSession = pglActionsMNE.downsampleEpochs.execute(session=session, downsampleFrequency=200)
if newSession: session = newSession

In [ ]:
session.saveSession()

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import pyarrow
from fsspec.implementations.local import LocalFileSystem

from pgl.pglSession import pglMNEIO, pglMNEVerifiedBacking


print(f"pandas: {pd.__version__}; PyArrow: {pyarrow.__version__}")

eventsID = session.mne.eventsID
assert isinstance(eventsID, pd.DataFrame), "session.mne.eventsID is not a DataFrame."

cases = [("actual", eventsID)]

for label, dtype in (("object-labels", object), ("string-labels", pd.StringDtype(na_value=np.nan))):
    frame = pd.DataFrame({"grouped": pd.Series(["blink", "stimulus", "other"], dtype=object), "code": [1, 2, 3]})
    frame.columns = pd.Index(["grouped", "code"], dtype=dtype, name="fields")
    cases.append((label, frame))

with TemporaryDirectory(prefix="pgl-eventsID-roundtrip-") as temporary:
    filesystem = LocalFileSystem(skip_instance_cache=True)

    for label, original in cases:
        for inferStrings in (False, True):
            destination = Path(temporary) / f"{label}-{inferStrings}"

            with pd.option_context("future.infer_string", inferStrings):
                bundle = pglMNEIO.saveEventsID(original, destination, filesystem=filesystem)
                payloadPath = str(Path(bundle["root"]) / bundle["entrypoint"])
                backing = pglMNEVerifiedBacking(filesystem, payloadPath, bundle["files"][0])
                restored = pglMNEIO.loadEventsID(backing)

            pd.testing.assert_frame_equal(original, restored, check_dtype=True, check_index_type=True, check_column_type=True, check_exact=True, check_categorical=True, check_flags=True)
            print(f"PASS: {label}, infer_string={inferStrings}")

print("All eventsID checks passed; temporary payloads removed.")

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import pyarrow
from fsspec.implementations.local import LocalFileSystem

from pgl.pglSession import pglMNEIO, pglMNEVerifiedBacking


print(f"pandas: {pd.__version__}; PyArrow: {pyarrow.__version__}")

eventsID = session.mne.eventsID
assert isinstance(eventsID, pd.DataFrame), "session.mne.eventsID is not a DataFrame."

with TemporaryDirectory(prefix="pgl-eventsID-roundtrip-") as temporary:
    filesystem = LocalFileSystem(skip_instance_cache=True)
    bundle = pglMNEIO.saveEventsID(eventsID, Path(temporary) / "eventsID", filesystem=filesystem)
    payloadPath = str(Path(bundle["root"]) / bundle["entrypoint"])
    backing = pglMNEVerifiedBacking(filesystem, payloadPath, bundle["files"][0])
    restoredEventsID = pglMNEIO.loadEventsID(backing)
    pd.testing.assert_frame_equal(eventsID, restoredEventsID, check_dtype=True, check_index_type=True, check_column_type=True, check_exact=True, check_categorical=True, check_flags=True)
    print(pd.DataFrame({"original": eventsID.dtypes, "restored": restoredEventsID.dtypes}))
    print("PASS: actual eventsID save/load preserves the compared values, dtypes, indexes, and flags.")

print("Temporary test payload removed.")

---

Compute NCSNR

---

In [ ]:
computeNCSNR = pglActionsMNE.computeNCSNR()
#computeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag",channelName="L213_bz-s115")
#computeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag")
computeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag",channelName="L213_bz-s115")


In [ ]:
newSession = computeNCSNR.run(session)
if newSession: session = newSession

---

look at spectrum

---

---

History

---

In [ ]:
session.history()

---

SCRATCH code

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pathEffects

channelName, peakTime, peakAmplitude = session.mne.evoked.get_peak(ch_type="mag", mode="abs", return_amplitude=True)
peakTime=0.250
with plt.rc_context({"figure.constrained_layout.use": False, "figure.autolayout": False}):
    fig = session.mne.evoked.plot_topomap(
        times=peakTime,
        ch_type="mag",
        average=0.02,
        sensors=True,
        show_names=True,
        show=False,
    )

    fig.set_size_inches(24, 16, forward=True)

    # ax.texts contains the channel-name labels added to the topomap.
    for ax in fig.axes:
        for text in ax.texts:
            text.set_color("white")
            text.set_fontsize(12)
            text.set_fontweight("bold")
            text.set_path_effects([
                pathEffects.Stroke(linewidth=2.5, foreground="black"),
                pathEffects.Normal(),
            ])

    fig.suptitle(f"Topomap at peak: {channelName} | {peakTime * 1000:.1f} ms", fontsize=18, fontweight="bold")
    plt.show()
print(session.mne.evoked.ch_names)

In [ ]:
session.mne.lookupSensor('L101')

Do spectrum on the evoked averages - take the mean of the 3000 trials in time domain and transform last
do it over 2 seconds - you should see nothing at 0.5Hz 

Compute how many seconds (in averages) and how many trials
Same yaxis
to look at sqrt(n) - compute 
Deflate the noise to compare to the number of trials

In [ ]:
%matplotlib qt
import mne
mne.viz.set_browser_backend("qt")
#session.mne.raw.plot(duration=20, n_channels=30, scalings="auto", block=True)
session.mne.raw.compute_psd().plot()

In [ ]:
session.mne.report.save("/Users/justin/Desktop/test.html", overwrite=True, open_browser=True)

In [ ]:
session.history()